In [36]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [3]:
load_dotenv()

True

In [28]:
model = ChatGoogleGenerativeAI(model= "models/gemini-3.1-flash-lite-preview")

In [33]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description = " Detailed feedback for essay")
    score: int = Field(description="score out of 10", ge=0, le=10)


In [34]:
structured_model = model.with_structured_output(EvaluationSchema,
    method = "function_calling"
)

In [31]:
essay = """
India’s Role in Artificial Intelligence

Artificial Intelligence (AI) is one of the most important technologies of the 21st century. It is changing the way people work, learn, communicate and solve problems. India, with its large population, strong IT sector and growing number of skilled professionals, has an important role to play in the development and use of AI.

India has already become a major centre for information technology and software services. Indian engineers and scientists are contributing to AI research and developing solutions for different sectors. The government is also promoting AI through initiatives such as the IndiaAI Mission, which aims to build AI infrastructure, skills and innovation in the country.

AI can help India address many important challenges. In agriculture, it can help farmers predict weather, identify crop diseases and improve productivity. In healthcare, AI can support early diagnosis and make medical services more accessible. In education, it can provide personalised learning and help students understand difficult subjects. AI can also improve public services, transport, banking and disaster management.

However, the growth of AI also brings challenges. There are concerns about job displacement, privacy, misinformation and the misuse of technology. India must therefore ensure that AI is developed in a responsible, safe and inclusive manner. Education and skill development are also necessary so that workers can adapt to changes brought by automation.

India can become a global leader in AI if it combines technology with its human resources and democratic values. It should focus not only on creating AI but also on using it to improve the lives of ordinary citizens.

In conclusion, India has a significant role in the global AI revolution. With skilled youth, technological capability and proper government support, India can use AI as a tool for economic growth, social development and better governance. The future of AI in India should be innovative, inclusive and responsible.
"""

In [35]:
prompt = f"Evaluate the essay and provide the feedback  and give the score out of 10 \n: {essay} "

response = structured_model.invoke(prompt)

response

EvaluationSchema(feedback='The essay provides a clear, well-structured, and balanced overview of India’s role in the AI landscape. \n\nStrengths:\n- Clarity and Structure: The essay flows logically from an introduction to potential applications, challenges, and a forward-looking conclusion. \n- Content: You effectively identified the key pillars—IT infrastructure, government initiatives (IndiaAI Mission), and societal impact (agriculture, healthcare, education).\n- Tone: The tone is professional, objective, and well-suited for an expository or informative piece.\n\nAreas for Improvement:\n- Depth of Analysis: While the essay covers the main points, it remains somewhat surface-level. For instance, mentioning specific types of AI (Generative AI, LLMs) or specific Indian start-ups/companies involved in the space would add technical depth and credibility.\n- Nuance in Challenges: The challenges section is correct but generic. You could elaborate on the \'digital divide\' in India—the risk 

In [38]:
#defie state
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [ ]:
def eval_language(state: UPSCState):

    prompt = f"Evaluate the langauge quality and provide the feedback  and give the score out of 10 \n: {state['essay']} "

    output = structured_model.invoke(prompt)

    return {
        "language_feedback": output.feedback,
        "individual_score": [output.score]
    }

In [40]:
def eval_analysis(state: UPSCState):

    prompt = f"Evaluate the depth of analysis and provide the feedback  and give the score out of 10 \n: {state['essay']} "

    output = structured_model.invoke(prompt)

    return {
        "analysis_feedback": output.feedback,
        "individual_score": [output.score]
    }

In [41]:
def eval_thought(state: UPSCState):

    prompt = f"Evaluate the clarity of thought and provide the feedback  and give the score out of 10 \n: {state['essay']} "

    output = structured_model.invoke(prompt)

    return {
        "clarity_feedback": output.feedback,
        "individual_score": [output.score]
    }

In [43]:
def final_eval(state: UPSCState):

    prompt = f" Based on the following feedbacks create a summarized feedback \n language_feeback = {state['language_feedback']} \n analysis_feedback = {state['analysis_feedback']}\n clarity_feedback = {state['clarity_feedback']} "

    overall_feedback = model.invoke(prompt).content

    avg_score = sum(state['individual_score']) / len(state['individual_score'])

    return {
        "overall_feedback" : overall_feedback,
        "avg_score" : avg_score
    }

In [47]:
graph = StateGraph(UPSCState)

#nodes
graph.add_node("eval_language", eval_language)
graph.add_node("eval_analysis", eval_analysis)
graph.add_node("eval_thought", eval_thought)
graph.add_node("final_eval", final_eval)

#edges
graph.add_edge(START, "eval_language")
graph.add_edge(START, "eval_analysis")
graph.add_edge(START, "eval_thought")

graph.add_edge("eval_language", "final_eval")
graph.add_edge("eval_analysis", "final_eval")
graph.add_edge("eval_thought", "final_eval")

graph.add_edge("final_eval", END)

workflow = graph.compile()



In [48]:
initial_state = { "essay" : essay }

final_state = workflow.invoke(initial_state)

final_state

{'essay': '\nIndia’s Role in Artificial Intelligence\n\nArtificial Intelligence (AI) is one of the most important technologies of the 21st century. It is changing the way people work, learn, communicate and solve problems. India, with its large population, strong IT sector and growing number of skilled professionals, has an important role to play in the development and use of AI.\n\nIndia has already become a major centre for information technology and software services. Indian engineers and scientists are contributing to AI research and developing solutions for different sectors. The government is also promoting AI through initiatives such as the IndiaAI Mission, which aims to build AI infrastructure, skills and innovation in the country.\n\nAI can help India address many important challenges. In agriculture, it can help farmers predict weather, identify crop diseases and improve productivity. In healthcare, AI can support early diagnosis and make medical services more accessible. In 